In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [24]:
root_dir = Path.cwd().parent
processed_dir = root_dir / "dataset" / "processed"
clf_train = pd.read_parquet(processed_dir / "clf_train_labeled_v1.parquet")
clf_val = pd.read_parquet(processed_dir / "clf_val_labeled_v1.parquet")

In [25]:
clf_val["predicted_intent"].value_counts()

predicted_intent
delivery_service_complaint      388
other                           268
delivery_late                   203
general_agent_escalation         85
device_and_technical_support     74
delivery_missing                 72
prime_subscription_issue         70
order_tracking_inquiry           68
order_cancellation               41
amazon_india_support             40
account_and_login_issues         38
amazon_pay_fintech               32
prime_billing_complaint          23
gift_card_problems               17
Name: count, dtype: int64

# 1) Trivial Baseline - 

## Always predicting majority class only

In [30]:
# 1. Clean out any empty rows from your dataframes
train_clean = clf_train.dropna(subset=['text', 'predicted_intent'])
val_clean = clf_val.dropna(subset=['text', 'predicted_intent'])

y_val_true = val_clean['predicted_intent'].values

# ========================================================
# METRIC 1: THE TRIVIAL BASELINE (Most Frequent Class)
# ========================================================
# Predict 'delivery_service_complaint' for every single validation row
majority_class = "delivery_service_complaint"
y_pred_trivial = [majority_class] * len(y_val_true)

print("="*60)
print("   TRIVIAL BASELINE REPORT (Majority Class Framework)   ")
print("="*60)
print(classification_report(y_val_true, y_pred_trivial, zero_division=0))

   TRIVIAL BASELINE REPORT (Majority Class Framework)   
                              precision    recall  f1-score   support

    account_and_login_issues       0.00      0.00      0.00        38
        amazon_india_support       0.00      0.00      0.00        40
          amazon_pay_fintech       0.00      0.00      0.00        32
               delivery_late       0.00      0.00      0.00       203
            delivery_missing       0.00      0.00      0.00        72
  delivery_service_complaint       0.27      1.00      0.43       388
device_and_technical_support       0.00      0.00      0.00        74
    general_agent_escalation       0.00      0.00      0.00        85
          gift_card_problems       0.00      0.00      0.00        17
          order_cancellation       0.00      0.00      0.00        41
      order_tracking_inquiry       0.00      0.00      0.00        68
                       other       0.00      0.00      0.00       268
     prime_billing_complaint    

# 2) Simple Baseline -

## Using TF-IDF and Logistic Regression

In [32]:
# ========================================================
# METRIC 2: THE SIMPLE BASELINE (TF-IDF + Logistic Regression)
# ========================================================
simple_baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, stop_words='english')),
    ('classifier', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42))
])

# Train instantly on your training data
simple_baseline_model.fit(train_clean['text'], train_clean['predicted_intent'])

# Predict against validation data
y_pred_simple = simple_baseline_model.predict(val_clean['text'])

print("="*60)
print("   SIMPLE BASELINE REPORT (TF-IDF + Logistic Regression)   ")
print("="*60)
print(classification_report(y_val_true, y_pred_simple, zero_division=0))

   SIMPLE BASELINE REPORT (TF-IDF + Logistic Regression)   
                              precision    recall  f1-score   support

    account_and_login_issues       0.75      0.79      0.77        38
        amazon_india_support       0.76      0.72      0.74        40
          amazon_pay_fintech       0.55      0.66      0.60        32
               delivery_late       0.60      0.58      0.59       203
            delivery_missing       0.36      0.58      0.45        72
  delivery_service_complaint       0.82      0.45      0.58       388
device_and_technical_support       0.69      0.89      0.78        74
    general_agent_escalation       0.46      0.74      0.57        85
          gift_card_problems       0.69      0.65      0.67        17
          order_cancellation       0.43      0.83      0.57        41
      order_tracking_inquiry       0.38      0.54      0.45        68
                       other       0.68      0.54      0.60       268
     prime_billing_complaint 